# SWE3050-41 Term Project: F1 Champion Prediction

성균관대학교 SKKU

Group 11: Nguyen Andy, UZMA NABEEHA BINTI SUFFIAN, 앙가락, 이나현



## Phase 1: Data Collection
### 1단계: 데이터 수집
Load raw CSV files from Kaggle (results, drivers, races, and constructors), then print the full dataset of F1 results from 1950 to 2024. After that, sort the results chronologically by year and races, then filter relevant (modern) data from 2014 to 2023.

Work by 앤디 Andy

In [1]:
import shutil

# delete old f1_cache folder
shutil.rmtree('/content/f1_cache', ignore_errors=True)
print("✅ Deleted f1_cache folder completely.")

✅ Deleted f1_cache folder completely.


In [2]:
# install fastf1 if not yet installed
!pip install fastf1

import os, fastf1

# re-initialise new fastf1 cache
os.makedirs('/content/f1_cache', exist_ok=True)
fastf1.Cache.enable_cache('/content/f1_cache')
print("🆕 New FastF1 cache initialized.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.0/123.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 5.2 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-1.0.2-cp312-cp312-linux_x86_64.whl size=15820 sha256=af2fc023196d5a0360a8b17dc29018f0b45c858f17e6f04814a7861d98c1393b
  Stored in directory: /root/.cache/pip/wheels/67/a6/40/eda0983e595bbf3841af96dbff2340be72dfac96796fc3d578
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.1.2
    Uninstalling msgpack-1.1.2:
      Successfully uninstalled msgpack-1.1.2
  Attempting uninstall: websocke

In [3]:
import pandas as pd

# 🏎️ STEP 1: LOAD RAW CSV FILES
print("📂 Loading Kaggle F1 dataset files...")

# upload these following files to Google Colab (in 'kaggle' folder)
kaggle_results = pd.read_csv('/kaggle/results.csv')
races = pd.read_csv('/kaggle/races.csv')[['raceId', 'year', 'name']]
drivers = pd.read_csv('/kaggle/drivers.csv')[['driverId', 'driverRef', 'surname']]
constructors = pd.read_csv('/kaggle/constructors.csv')[['constructorId', 'name']]

print("✅ All source files loaded successfully.")

# 🧩 STEP 2: MERGE INTO ONE COMPLETE DATAFRAME
merged = (
    kaggle_results
    .merge(races, on='raceId')
    .merge(drivers, on='driverId')
    .merge(constructors, on='constructorId')
)

# rename columns for clarity
merged = merged.rename(columns={
    'year': 'Year',
    'name_x': 'Race',
    'surname': 'Driver',
    'name_y': 'Constructor',
    'positionOrder': 'Position',
    'points': 'Points'
})

# save merged dataset
merged.to_csv('/content/formula1_results_1950_2024.csv', index=False)
print("💾 Merged dataset saved as formula1_results_1950_2024.csv")

# 🧹 STEP 3: CLEAN AND SORT DATA
print("🧹 Cleaning and sorting dataset...")

df = pd.read_csv('/content/formula1_results_1950_2024.csv')

# select and reorder relevant columns
df = df[['Year', 'Race', 'Driver', 'Constructor', 'Position', 'Points']]

# drop missing / invalid rows
df = df.dropna(subset=['Year', 'Race', 'Driver', 'Constructor', 'Points'])
df = df[df['Position'] != '\n']

# ensure numeric columns have proper data types
df['Year'] = df['Year'].astype(int)
df['Position'] = df['Position'].astype(int)
df['Points'] = df['Points'].astype(float)

# sort chronologically
df = df.sort_values(by=['Year', 'Race']).reset_index(drop=True)

print(f"✅ Cleaned dataset shape: {df.shape}")

# save the clean version
df.to_csv('/content/formula1_results_cleaned.csv', index=False)
print("💾 Saved clean dataset as formula1_results_cleaned.csv")

# 🎯 STEP 4: FILTER MODERN ERA (2014–2023)
# skip 2024 due to partial/incomplete results
df_filtered = df[df['Year'].between(2014, 2023)]
df_filtered.to_csv('/content/f1_results_2014_2023.csv', index=False)
print(f"🏁 Saved modern-era filtered dataset (2014–2023) → {len(df_filtered)} rows")

📂 Loading Kaggle F1 dataset files...
✅ All source files loaded successfully.
💾 Merged dataset saved as formula1_results_1950_2024.csv
🧹 Cleaning and sorting dataset...
✅ Cleaned dataset shape: (26759, 6)
💾 Saved clean dataset as formula1_results_cleaned.csv
🏁 Saved modern-era filtered dataset (2014–2023) → 4147 rows


## Phase 2: Feature Engineering
### 2단계: 기능 엔지니어링

Transforming clean race-level data into season-level driver statistics, ready for model training (for predicting champions).

Work by 앤디 Andy

In [4]:
# 🧮 FEATURE ENGINEERING: CREATE DRIVER-SEASON LEVEL DATA

import pandas as pd

# load cleaned race-level dataset
# NOTE: make sure to upload "f1_results_2014_2023.csv" to Google Colab so this script can read the file
df = pd.read_csv('/content/f1_results_2014_2023.csv')

# ensure correct datatypes
df['Points'] = df['Points'].astype(float)
df['Position'] = df['Position'].astype(int)

# aggregate key performance features per driver per season
features = (
    df.groupby(['Year', 'Driver'])
      .agg(
          Total_Points=('Points', 'sum'),
          Wins=('Position', lambda x: (x == 1).sum()),
          Podiums=('Position', lambda x: (x <= 3).sum()),
          Top10s=('Position', lambda x: (x <= 10).sum()),
          Races_Entered=('Race', 'count'),
          Avg_Position=('Position', 'mean')
      )
      .reset_index()
)

# add a feature for consistency (lower = better)
features['Position_STD'] = (
    df.groupby(['Year', 'Driver'])['Position'].std().reset_index(drop=True)
)

# find the champion for each season (label = 1 if champion, else 0)
champions = (
    features.loc[features.groupby('Year')['Total_Points'].idxmax(), ['Year', 'Driver']]
    .assign(Champion=1)
)

# merge back to create binary label column
features = features.merge(champions, on=['Year', 'Driver'], how='left')
features['Champion'] = features['Champion'].fillna(0).astype(int)

# sort and save
features = features.sort_values(['Year', 'Total_Points'], ascending=[True, False])
features.to_csv('/content/f1_driver_features_2014_2023.csv', index=False)

print("✅ Feature engineering complete!")
print("📄 Saved: f1_driver_features_2014_2023.csv")
print(f"Rows: {len(features)}, Columns: {features.shape[1]}")
print("\nSample preview:")
print(features.head(10))

✅ Feature engineering complete!
📄 Saved: f1_driver_features_2014_2023.csv
Rows: 223, Columns: 10

Sample preview:
    Year      Driver  Total_Points  Wins  Podiums  Top10s  Races_Entered  \
8   2014    Hamilton         384.0    11       16      16             19   
18  2014     Rosberg         317.0     5       15      16             19   
17  2014   Ricciardo         238.0     3        8      16             19   
2   2014      Bottas         186.0     0        6      17             19   
23  2014      Vettel         167.0     0        4      16             19   
0   2014      Alonso         161.0     0        2      17             19   
15  2014       Massa         134.0     0        3      11             19   
3   2014      Button         126.0     0        1      13             19   
9   2014  Hülkenberg          96.0     0        0      15             19   
16  2014       Pérez          59.0     0        1      12             19   

    Avg_Position  Position_STD  Champion  
8     

## Phase 3: Model Training & F1 Champion Prediction Results
### 3단계: 모델 학습 및 F1 챔피언 예측 결과

Work by Uzma, 이나현
#### 1: Data Preparation & Time-Based Split

In [5]:
from google.colab import files
import pandas as pd

# load the driver-season feature dataset
df = pd.read_csv('/content/f1_driver_features_2014_2023.csv')

# fill missing std with mean
df['Position_STD'] = df['Position_STD'].fillna(df['Position_STD'].mean())

# define feature columns
feature_cols = [
    'Total_Points', 'Wins', 'Podiums', 'Top10s',
    'Races_Entered', 'Avg_Position', 'Position_STD'
]

# time-based split: no shuffling, no leakage
train_years = list(range(2014, 2020))  # 2014–2019
val_years   = [2020, 2021]
test_years  = [2022, 2023]

train_df = df[df['Year'].isin(train_years)].copy()
val_df   = df[df['Year'].isin(val_years)].copy()
test_df  = df[df['Year'].isin(test_years)].copy()

X_train = train_df[feature_cols]
y_train = train_df['Champion']

X_val   = val_df[feature_cols]
y_val   = val_df['Champion']

X_test  = test_df[feature_cols]
y_test  = test_df['Champion']

print("train years:", sorted(train_df['Year'].unique()), "shape:", X_train.shape)
print("val years:  ", sorted(val_df['Year'].unique()), "shape:", X_val.shape)
print("test years: ", sorted(test_df['Year'].unique()), "shape:", X_test.shape)

train years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)] shape: (135, 7)
val years:   [np.int64(2020), np.int64(2021)] shape: (44, 7)
test years:  [np.int64(2022), np.int64(2023)] shape: (44, 7)


#### 2. Define Models (Logistic Regression, Decision Tree, Random Forests, SVM) with Scaling & Class Weights

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# add scaling and class_weight
models = {
    "Logistic Regression": Pipeline([
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Support Vector Machine (SVM)": Pipeline([
        ('scale', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
    ])
}


#### 3. Train & Choose Best Model

In [7]:
from sklearn.metrics import classification_report

best_models = {}
val_scores = {}

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    report = classification_report(y_val, y_val_pred, output_dict=True)
    f1_champion = report['1']['f1-score']
    val_scores[name] = f1_champion
    best_models[name] = model

    print(classification_report(y_val, y_val_pred, digits=4))
    print(f"F1 score for champion class (1): {f1_champion:.4f}")

# select best model based on F1 for champion class
best_name = max(val_scores, key=val_scores.get)
best_model = best_models[best_name]

print(f"Best model selected: {best_name}")



=== Training Logistic Regression ===
              precision    recall  f1-score   support

           0     1.0000    0.9762    0.9880        42
           1     0.6667    1.0000    0.8000         2

    accuracy                         0.9773        44
   macro avg     0.8333    0.9881    0.8940        44
weighted avg     0.9848    0.9773    0.9794        44

F1 score for champion class (1): 0.8000

=== Training Decision Tree ===
              precision    recall  f1-score   support

           0     1.0000    0.9762    0.9880        42
           1     0.6667    1.0000    0.8000         2

    accuracy                         0.9773        44
   macro avg     0.8333    0.9881    0.8940        44
weighted avg     0.9848    0.9773    0.9794        44

F1 score for champion class (1): 0.8000

=== Training Random Forest ===
              precision    recall  f1-score   support

           0     1.0000    0.9762    0.9880        42
           1     0.6667    1.0000    0.8000         2



#### 4. Final Classification Evaluation on Test Set

In [8]:
from sklearn.metrics import accuracy_score, classification_report

y_test_pred = best_model.predict(X_test)

print("=== Test Set Classification ===")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=4))

=== Test Set Classification ===
Accuracy: 1.0
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        42
           1     1.0000    1.0000    1.0000         2

    accuracy                         1.0000        44
   macro avg     1.0000    1.0000    1.0000        44
weighted avg     1.0000    1.0000    1.0000        44



#### 5. Ranking Evaluation & Champion Prediction

In [9]:
# add predicted probabilities for test set
test_df = test_df.copy()
test_df['pred_proba'] = best_model.predict_proba(X_test)[:, 1]

def ranking_metrics(df_season, top_k=3):
    ranked = df_season.sort_values('pred_proba', ascending=False)
    champion_idx = ranked['Champion'].values.argmax()  # index of first '1'
    hit1 = int(champion_idx == 0)
    hitk = int(champion_idx < top_k)
    rank = champion_idx + 1
    return rank, hit1, hitk, ranked

results = []
for year, grp in test_df.groupby('Year'):
    rank, hit1, hit3, ranked = ranking_metrics(grp, top_k=3)
    results.append({
        'Year': year,
        'Champion_Rank': rank,
        'Hit@1': hit1,
        'Hit@3': hit3
    })

ranking_df = pd.DataFrame(results).sort_values('Year')
print("=== Champion Rank per Test Season ===")
print(ranking_df)
print("\nOverall Hit@1:", ranking_df['Hit@1'].mean())
print("Overall Hit@3:", ranking_df['Hit@3'].mean())

=== Champion Rank per Test Season ===
   Year  Champion_Rank  Hit@1  Hit@3
0  2022              1      1      1
1  2023              1      1      1

Overall Hit@1: 1.0
Overall Hit@3: 1.0


#### 6. Predicted vs. True Champion

In [10]:
predicted_champions = []

for year, grp in test_df.groupby('Year'):
    ranked = grp.sort_values('pred_proba', ascending=False)
    predicted_driver = ranked.iloc[0]['Driver']
    true_driver = ranked.loc[ranked['Champion'] == 1, 'Driver'].iloc[0]
    predicted_champions.append({
        'Year': year,
        'Predicted Champion': predicted_driver,
        'True Champion': true_driver,
        'Correct?': predicted_driver == true_driver
    })

predicted_df = pd.DataFrame(predicted_champions).sort_values('Year')
print("=== Predicted vs True Champions (Test Years) ===")
print(predicted_df)

=== Predicted vs True Champions (Test Years) ===
   Year Predicted Champion True Champion  Correct?
0  2022         Verstappen    Verstappen      True
1  2023         Verstappen    Verstappen      True


#### 7. 🥉 Top 3 Ranked Drivers for Each Season

In [11]:
top3_list = []

for year, grp in test_df.groupby('Year'):
    ranked = grp.sort_values('pred_proba', ascending=False).head(3)
    top3_list.append({
        'Year': year,
        '1st': ranked.iloc[0]['Driver'],
        '2nd': ranked.iloc[1]['Driver'],
        '3rd': ranked.iloc[2]['Driver']
    })

top3_df = pd.DataFrame(top3_list).sort_values('Year')
top3_df

,Year,1st,2nd,3rd
0,2022,Verstappen,Leclerc,Pérez
1,2023,Verstappen,Pérez,Hamilton


#### 8. 🏁 F1 Prediction Interaction Dashboard

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML
import pandas as pd

# ===============================
# 1. Season / Driver Prediction Function
# ===============================

def get_season_prediction(year: int):
    """
    For the given season (year):
    - Compute pred_proba using test_df + X_test
    - Sort all drivers by predicted probability
    - Return model’s top prediction, true champion, and Top3
    """
    df_with_proba = test_df.copy()
    df_with_proba = df_with_proba.assign(
        pred_proba = best_model.predict_proba(X_test)[:, 1]
    )

    season_df = df_with_proba[df_with_proba['Year'] == year].copy()
    ranked = season_df.sort_values('pred_proba', ascending=False)

    predicted_champion = ranked.iloc[0]['Driver']
    true_champion = ranked.loc[ranked['Champion'] == 1, 'Driver'].iloc[0]
    top3 = ranked.head(3).reset_index(drop=True)

    return season_df, ranked, predicted_champion, true_champion, top3


# ===============================
# 2. UI Widgets
# ===============================

available_years = sorted(test_df['Year'].unique())
year_options = [y for y in available_years if y in (2022, 2023)]

year_dropdown = widgets.Dropdown(
    options=year_options,
    value=year_options[0],
    description='Season:',
    style={'description_width': '80px'}
)

driver_dropdown = widgets.Dropdown(
    options=[],
    description='Your pick:',
    style={'description_width': '80px'}
)

run_button = widgets.Button(
    description='🏁 Run Prediction',
    button_style='success',
    icon='play'
)

btn_why = widgets.Button(
    description='🤔 Why this prediction?',
    button_style='info',
    disabled=True
)
btn_top3 = widgets.Button(
    description='🏎 View Top 3 Predictions',
    button_style='primary',
    disabled=True
)
btn_reset = widgets.Button(
    description='🔁 Reset',
    button_style='warning',
    disabled=True
)

result_output = widgets.Output()

state = {
    'year': None,
    'user_pick': None,
    'predicted': None,
    'true': None,
    'top3': None,
    'ranked': None
}


# ===============================
# 3. Update Driver List by Season
# ===============================

def update_driver_options(year):
    drivers = sorted(test_df[test_df['Year'] == year]['Driver'].unique())
    driver_dropdown.options = drivers
    if drivers:
        driver_dropdown.value = drivers[0]

def on_year_change(change):
    if change['name'] == 'value':
        update_driver_options(change['new'])

year_dropdown.observe(on_year_change, names='value')
update_driver_options(year_dropdown.value)


# ===============================
# 4. Rendering Functions
# ===============================

def render_main_result():
    """Display user vs model vs actual champion"""
    with result_output:
        clear_output()

        year = state['year']
        user_pick = state['user_pick']
        predicted = state['predicted']
        true = state['true']

        correct_model = (predicted == true)
        correct_user = (user_pick == true)

        model_color = "#4CAF50" if correct_model else "#F44336"
        model_emoji = "✅" if correct_model else "❌"
        user_color = "#4CAF50" if correct_user else "#F44336"
        user_emoji = "✅" if correct_user else "❌"

        html_card = f"""
        <div style="
            background-color: #2b2b2b;
            color: #f5f5f5;
            border-radius: 14px;
            box-shadow: 0 4px 18px rgba(0,0,0,0.4);
            padding: 24px;
            max-width: 700px;
            margin: 15px auto;
            font-family: 'Segoe UI', sans-serif;
            text-align: center;
        ">
            <h3 style="color:#80d8ff; margin-top:0;">🚦 F1 {year} Season – Champion Prediction</h3>
            <p style="font-size:15px; opacity:0.9;">
                Below shows your prediction, the model’s prediction, and the actual champion.
            </p>
            <div style="display:flex; justify-content:space-around; margin-top:18px; flex-wrap:wrap;">
                <div style="min-width:200px; margin:8px 0;">
                    <h4 style="margin-bottom:6px;">👤 Your Pick</h4>
                    <div style="font-size:16px;">
                        <span style="color:#ffd54f;">{user_pick}</span><br>
                        <span style="color:{user_color}; font-size:18px;">{user_emoji}</span>
                        <span style="font-size:14px;"> ({'Matched' if correct_user else 'Not Matched'})</span>
                    </div>
                </div>
                <div style="min-width:200px; margin:8px 0;">
                    <h4 style="margin-bottom:6px;">🤖 Model Prediction</h4>
                    <div style="font-size:16px;">
                        <span style="color:#4fc3f7;">{predicted}</span><br>
                        <span style="color:{model_color}; font-size:18px;">{model_emoji}</span>
                        <span style="font-size:14px;"> ({'Matched' if correct_model else 'Not Matched'})</span>
                    </div>
                </div>
                <div style="min-width:200px; margin:8px 0;">
                    <h4 style="margin-bottom:6px;">🏆 Actual Champion</h4>
                    <div style="font-size:16px;">
                        <span style="color:#ce93d8;">{true}</span>
                    </div>
                </div>
            </div>
            <p style="margin-top:18px; font-size:13px; opacity:0.8;">
                Click the buttons below to explore <b>why</b> the model made this prediction
                and to view <b>Top 3 Predicted Drivers</b>.
            </p>
        </div>
        """
        display(DHTML(html_card))
        display(widgets.HBox([btn_why, btn_top3, btn_reset]))


def render_why_explanation():
    """Explain why the model made this prediction"""
    with result_output:
        clear_output()

        top3 = state['top3']

        html_title = """
        <div style="
            background-color:#263238;
            color:#e0f7fa;
            border-radius:10px;
            padding:16px;
            max-width:700px;
            margin:10px auto;
            font-family:'Segoe UI', sans-serif;
        ">
            <h3 style="margin:0 0 8px 0;">🤔 Why did the model predict this?</h3>
            <p style="margin:0; font-size:14px; opacity:0.9;">
                Here are the main features used by the model for the top 3 drivers.
                Generally, higher <b>Total_Points</b>, <b>Wins</b>, and <b>Podiums</b>
                and lower <b>Avg_Position</b> lead to higher championship probability.
            </p>
        </div>
        """
        display(DHTML(html_title))

        cols = ['Driver', 'Champion', 'pred_proba'] + [
            c for c in feature_cols if c in top3.columns
        ]
        df_show = top3[cols].copy()
        df_show = df_show.rename(columns={
            'Driver': 'Driver',
            'Champion': 'Is_Real_Champion',
            'pred_proba': 'Predicted_Champion_Prob'
        })

        display(df_show.style.format({
            'Predicted_Champion_Prob': '{:.3f}',
            'Avg_Position': '{:.2f}',
            'Position_STD': '{:.2f}'
        }))

        display(widgets.HBox([btn_why, btn_top3, btn_reset]))


def render_top3_view():
    """Show model vs actual ranks for Top 3"""
    with result_output:
        clear_output()

        ranked = state['ranked'].copy()
        year = state['year']

        ranked["Predicted_Rank"] = ranked["pred_proba"].rank(
            ascending=False, method="min"
        ).astype(int)

        if "Total_Points" in ranked.columns:
            ranked["Actual_Rank"] = ranked["Total_Points"].rank(
                ascending=False, method="min"
            ).astype(int)
        else:
            ranked["Actual_Rank"] = ranked["Champion"].apply(
                lambda x: 1 if x == 1 else None
            )

        top3 = ranked.nsmallest(3, "Predicted_Rank")[
            ["Driver", "Predicted_Rank", "Actual_Rank", "pred_proba", "Champion"]
        ]

        html_block = f"""
        <div style="
            background-color:#1c1f26;
            color:#eceff1;
            border-radius:10px;
            padding:18px;
            max-width:650px;
            margin:10px auto;
            font-family:'Segoe UI', sans-serif;
        ">
            <h3 style="margin-top:0;">🏎 Top 3 Predicted Drivers – {year}</h3>
            <table style="width:100%; border-collapse:collapse; font-size:14px; text-align:center;">
                <thead style="background-color:#37474f;">
                    <tr>
                        <th>Rank</th>
                        <th>Driver</th>
                        <th>Model Prob.</th>
                        <th>Predicted Rank</th>
                        <th>Actual Rank</th>
                        <th>Champion?</th>
                    </tr>
                </thead>
                <tbody>
        """

        for display_rank, (_, row) in enumerate(top3.iterrows(), start=1):
            is_champ = "🏆" if row["Champion"] == 1 else ""
            actual_rank_str = "-" if pd.isna(row["Actual_Rank"]) else int(row["Actual_Rank"])
            html_block += f"""
                <tr style="background-color:{'#263238' if display_rank%2==1 else '#2f3439'};">
                    <td>{display_rank}</td>
                    <td>{row['Driver']}</td>
                    <td>{row['pred_proba']:.3f}</td>
                    <td>{row['Predicted_Rank']}</td>
                    <td>{actual_rank_str}</td>
                    <td>{is_champ}</td>
                </tr>
            """

        html_block += """
                </tbody>
            </table>
            <p style="font-size:12px; opacity:0.8; margin-top:8px;">
                * Model Prob. = Predicted probability of being the champion.<br>
                * Predicted Rank = Ranking based on predicted probability.<br>
                * Actual Rank = True season rank based on total points.
            </p>
        </div>
        """
        display(DHTML(html_block))
        display(widgets.HBox([btn_why, btn_top3, btn_reset]))


def reset_view():
    with result_output:
        clear_output()
        display(DHTML("""
        <div style="
            background-color:#263238;
            color:#eceff1;
            border-radius:8px;
            padding:16px;
            max-width:600px;
            margin:10px auto;
            font-family:'Segoe UI', sans-serif;
            font-size:14px;
        ">
            Select a season and your predicted champion, then click
            <b>🏁 Run Prediction</b> to start.
        </div>
        """))
    btn_why.disabled = True
    btn_top3.disabled = True
    btn_reset.disabled = True


# ===============================
# 5. Event Handlers
# ===============================

def on_run_clicked(b):
    year = year_dropdown.value
    user_pick = driver_dropdown.value

    season_df, ranked, predicted, true, top3 = get_season_prediction(year)

    state['year'] = year
    state['user_pick'] = user_pick
    state['predicted'] = predicted
    state['true'] = true
    state['top3'] = top3
    state['ranked'] = ranked

    btn_why.disabled = False
    btn_top3.disabled = False
    btn_reset.disabled = False

    render_main_result()

def on_why_clicked(b):
    if state['top3'] is not None:
        render_why_explanation()

def on_top3_clicked(b):
    if state['top3'] is not None:
        render_top3_view()

def on_reset_clicked(b):
    reset_view()

run_button.on_click(on_run_clicked)
btn_why.on_click(on_why_clicked)
btn_top3.on_click(on_top3_clicked)
btn_reset.on_click(on_reset_clicked)


# ===============================
# 6. Display UI
# ===============================

controls_box = widgets.VBox([
    widgets.HTML(
        "<h3 style='font-family:Segoe UI; margin-bottom:4px;'>🎮 F1 Champion Prediction – Interactive Dashboard</h3>"
        "<p style='font-size:13px; margin-top:0; color:#607d8b;'>"
        "Select a season and your predicted champion driver, then click "
        "<b>Run Prediction</b> to compare your guess with the model and actual results.</p>"
    ),
    widgets.HBox([year_dropdown, driver_dropdown, run_button])
])

reset_view()
ui = widgets.VBox([controls_box, result_output])
display(ui)
